This notebook fixes the raw order JSON format, parses it, and creates the silver orders_json table.

In [0]:

df = spark.table('gizmobox.bronze.py_orders')
display(df)

In [0]:
from pyspark.sql.functions import *

In [0]:
df_fixed_orders = (
    df.select(
        regexp_replace('value', '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "$1"').alias ('fixed_value')
    )
)
display(df_fixed_orders)

In [0]:
df_with_schema =(
    df_fixed_orders.select(
        schema_of_json(col('fixed_value').alias('schema')
    )
))
display(df_with_schema.limit(1))

In [0]:
order_schema = '''STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'''

In [0]:
df_json_orders = (
    df_fixed_orders.select(
        from_json(col('fixed_value'), order_schema).alias('json_values')
    )
)
display(df_json_orders)

In [0]:
df_json_orders.selectExpr('json_values as json_value').write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gizmobox.silver.py_orders_json')